Covers → vectors (cover_pipeline).

15.071 Project 1, cover block **V**. Aligned with the team repo `godspeedhuang/royalroad-breakout`.

How to run:
1. Put this file on Drive at `RR_project1/cover_v2/cover_pipeline.ipynb`, open it in Colab, and pick a **T4 GPU** runtime.
2. `DRY_RUN` below is the only switch. Start with `True` for a trial run (300 serials, about 10 minutes); once it looks right, set it to `False` for the full run.
3. Menu: Runtime → Run all. If the session disconnects, Run all again; finished steps are skipped automatically.

Outputs go to `RR_project1/cover_v2/dry_run/` or `full/` on Drive. This notebook trains no model, only runs self-checks; 2023 (test year) labels are not used in any statistic.

In [1]:
DRY_RUN = False  # True = trial run (300 serials, written to dry_run/); False = full run (all serials, written to full/)

Step 1: environment.

In [2]:
import os, sys, io, re, json, time, shutil, subprocess, datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import threading

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

try:
    import imagehash
except ImportError:
    _pip("imagehash")
    import imagehash

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
import torch
import matplotlib.pyplot as plt
from PIL import Image

# ---- Fixed settings (section 2 of the task spec; do not change) ----
EXPECTED_N = int(os.environ.get("COVER_EXPECTED_N", 45245))   # expected number of rows in the book list
HF_REPO, HF_FILE = "syuanbo/royalroad-breakout", "embedding/out/books_meta.parquet"
SEED = 0
DRY_N = 300
IMG_SIZE = 384
N_THREADS = 16
TIMEOUT = 20
PART_SIZE = 100 if DRY_RUN else 2000
TRAIN_YEARS, VAL_YEAR, TEST_YEAR, RECENT_YEAR = (2017, 2021), 2022, 2023, 2024
MODELS = {"resnet50": 2048, "clip_b32": 512, "clip_l14": 768, "dinov2_base": 768}
STATUSES = ["no_custom_cover", "no_timestamp", "after_ch3", "download_failed", "image_changed", "valid"]
USER_AGENT = "Mozilla/5.0 (X11; Linux x86_64) cover-research/1.0 (MIT 15.071 course project)"

# ---- Folders (section 3 of the task spec) ----
if os.environ.get("COVER_LOCAL_ROOT"):            # set this environment variable to the root folder when running on a local machine
    ROOT = Path(os.environ["COVER_LOCAL_ROOT"])
else:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive")
    assert (ROOT / "RR_project1").exists(), "No RR_project1 folder on Drive. Make sure you are signed in to the Google account that holds it."

BASE = ROOT / "RR_project1" / "cover_v2"
RUN = BASE / ("dry_run" if DRY_RUN else "full")
D_IN, D_ST, D_IMG, D_EMB, D_EXP = RUN / "00_input", RUN / "01_status", RUN / "02_images", RUN / "03_embeddings", RUN / "04_explore"
D_PARTS = D_IMG / "_download_parts"
D_GH = RUN / "05_for_github"
D_OUT, D_FIG, D_LOG = D_GH / "out", D_GH / "out" / "figs", RUN / "logs"
for d in [D_IN, D_ST, D_IMG, D_PARTS, D_EMB, D_EXP, D_OUT, D_FIG, D_LOG]:
    d.mkdir(parents=True, exist_ok=True)

P_META = D_IN / "books_meta.parquet"
P_STATUS = D_ST / "cover_status.parquet"
P_SPOT = D_ST / "spot_check.csv"
P_IMG = D_IMG / "cover_images_384.parquet"
P_EMB = {m: D_EMB / f"cover_emb_{m}.parquet" for m in MODELS}
P_STATS = D_EMB / "cover_stats.parquet"
P_CLUST = D_EXP / "cover_clusters.parquet"

# ---- Logging and progress ----
T0 = time.time()
PROGRESS = {"run": "dry_run" if DRY_RUN else "full", "stage": "", "done": 0, "total": 0, "errors": 0,
            "stages_done": [], "finished": False, "failed": False, "message": ""}

def _now():
    return datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

def log(msg):
    line = f"[{_now()}] {msg}"
    print(line, flush=True)
    with open(D_LOG / "run_log.txt", "a", encoding="utf-8") as f:
        f.write(line + "\n")

def progress(stage=None, done=None, total=None, errors=None, **kw):
    if stage is not None: PROGRESS["stage"] = stage
    if done is not None: PROGRESS["done"] = int(done)
    if total is not None: PROGRESS["total"] = int(total)
    if errors is not None: PROGRESS["errors"] = int(errors)
    PROGRESS.update(kw)
    PROGRESS["elapsed_min"] = round((time.time() - T0) / 60, 1)
    PROGRESS["updated"] = _now()
    (D_LOG / "progress.json").write_text(json.dumps(PROGRESS, ensure_ascii=False, indent=1), encoding="utf-8")

def stage_start(name, total=0):
    log(f"=== start: {name} ===")
    progress(stage=name, done=0, total=total, errors=0)
    return time.time()

def stage_end(name, t_start, note=""):
    if name not in PROGRESS["stages_done"]:
        PROGRESS["stages_done"].append(name)
    progress()
    log(f"=== done: {name} ({(time.time() - t_start) / 60:.1f} min) {note} ===")

def check(cond, step, expected, actual):
    """Assertion. Stops on failure and says which step failed, what was expected and what was found."""
    if not cond:
        msg = f"[CHECK FAILED] step: {step} | expected: {expected} | actual: {actual}"
        log(msg)
        progress(failed=True, message=msg)
        raise RuntimeError(msg)

# ---- Resume: each step has a version number. Skip if the output exists with the same version; redoing a step also redoes the steps after it ----
STEP_VER = {"s4_download": 1, "s5_status": 3, "s6_resnet50": 1, "s6_clip_b32": 1, "s6_clip_l14": 1,
            "s6_dinov2_base": 1, "s7_stats": 1, "s8_selfcheck": 1, "s9_explore": 3}
ORDER = list(STEP_VER)
P_VERS = D_LOG / "step_versions.json"

def _vers():
    return json.loads(P_VERS.read_text()) if P_VERS.exists() else {}

def need(step, outputs):
    v = _vers()
    if v.get(step) == STEP_VER[step] and all(Path(o).exists() for o in outputs):
        log(f"{step}: output exists, skipping")
        return False
    # When this step is redone, later steps that depend on it are redone too (embeddings and visual stats depend only on the images, not on the status table)
    after = {"s4_download": ORDER[1:], "s7_stats": ["s8_selfcheck"], "s8_selfcheck": [], "s9_explore": []}
    for s in [step] + after.get(step, ["s8_selfcheck", "s9_explore"]):
        v.pop(s, None)
    P_VERS.write_text(json.dumps(v))
    return True

def mark(step):
    v = _vers()
    v[step] = STEP_VER[step]
    P_VERS.write_text(json.dumps(v))

# ---- Small file helpers ----
def save_df(df, path):
    tmp = str(path) + ".tmp"
    df.to_parquet(tmp, index=False)
    os.replace(tmp, path)

def load_df(path, columns=None):
    return pd.read_parquet(path, columns=columns)

def list_columns(path):
    return list(pq.read_schema(path).names)

def write_emb(path, ids, E):
    """fiction_id + emb (list of float32)"""
    E = np.ascontiguousarray(E, dtype=np.float32)
    n, d = E.shape
    emb = pa.ListArray.from_arrays(pa.array(np.arange(0, (n + 1) * d, d, dtype=np.int32)), pa.array(E.ravel()))
    tmp = str(path) + ".tmp"
    pq.write_table(pa.table({"fiction_id": pa.array(np.asarray(ids)), "emb": emb}), tmp, row_group_size=5000)
    os.replace(tmp, path)

def read_emb(path):
    t = pq.read_table(path)
    ids = t.column("fiction_id").to_numpy()
    E = t.column("emb").combine_chunks().flatten().to_numpy().reshape(len(ids), -1).astype(np.float32)
    return ids, E

def write_images(path, part_paths):
    """Merge the successful images from the download parts into one file. Returns the number of images."""
    tmp, writer, n = str(path) + ".tmp", None, 0
    for p in part_paths:
        d = load_df(p, ["fiction_id", "jpg"])
        d = d[d.jpg.notna()]
        if len(d) == 0:
            continue
        t = pa.Table.from_pandas(d, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(tmp, t.schema)
        writer.write_table(t.cast(writer.schema))
        n += len(d)
    if writer is None:
        return 0
    writer.close()
    os.replace(tmp, path)
    return n

_IMG_CACHE = {}
def read_images():
    """Returns (array of fiction_id, list of JPEG bytes). Read only once."""
    if not _IMG_CACHE:
        t = pq.read_table(P_IMG)
        _IMG_CACHE["ids"] = t.column("fiction_id").to_numpy()
        _IMG_CACHE["jpgs"] = t.column("jpg").to_pylist()
    return _IMG_CACHE["ids"], _IMG_CACHE["jpgs"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
log(f"Mode: {'trial DRY_RUN' if DRY_RUN else 'full run FULL'} | output folder: {RUN} | device: {DEVICE}")
if DEVICE == "cpu" and not DRY_RUN:
    log("Warning: no GPU detected. Pick a T4 GPU under Runtime → Change runtime type, or embedding extraction will be very slow.")
progress(stage="Step 1 environment")

Mounted at /content/drive
[2026-11-14 22:00:32 UTC] Mode: full run FULL | output folder: /content/drive/MyDrive/RR_project1/cover_v2/full | device: cuda


Step 2: read the book list.

In [3]:
_t = stage_start("Step 2 read book list")
if not P_META.exists():
    from huggingface_hub import hf_hub_download
    _p = hf_hub_download(HF_REPO, HF_FILE, repo_type="dataset")
    shutil.copy(_p, str(P_META) + ".tmp")
    os.replace(str(P_META) + ".tmp", P_META)
    log("Downloaded books_meta.parquet from Hugging Face")

_cols = list_columns(P_META)
print("All fields in the book list:", _cols)
_need = ["fiction_id", "cover_url", "first_release", "year", "quarter", "top10", "author_id"]
check(all(c in _cols for c in _need), "Step 2 read book list", f"book list has fields {_need}", f"missing {[c for c in _need if c not in _cols]}")
check("first3_last" in _cols or "first3_span_days" in _cols, "Step 2 read book list",
      "book list has first3_last or first3_span_days", f"neither is present; fields are {_cols}")
_extra = ["first3_last"] if "first3_last" in _cols else ["first3_span_days"]
books_all = load_df(P_META, _need + _extra)

books_all["first_release"] = pd.to_datetime(books_all["first_release"], utc=True)
if "first3_last" in books_all:
    books_all["first3_last"] = pd.to_datetime(books_all["first3_last"], utc=True)
    FIRST3_SOURCE = "first3_last field in books_meta"
else:   # the team's 01_prepare.py does not keep first3_last; rebuild it from first_release + first3_span_days
    books_all["first3_last"] = books_all["first_release"] + pd.to_timedelta(books_all["first3_span_days"].astype(float), unit="D")
    FIRST3_SOURCE = "first_release + first3_span_days (books_meta has no first3_last)"
books_all["year"] = books_all["year"].astype(int)
books_all["top10"] = books_all["top10"].astype(bool)
books_all = books_all[_need + ["first3_last"]]

check(len(books_all) == EXPECTED_N, "Step 2 read book list", f"book list has {EXPECTED_N} rows", f"{len(books_all)} rows")
check(books_all.fiction_id.is_unique, "Step 2 read book list", "no duplicate fiction_id", f"{int(books_all.fiction_id.duplicated().sum())} duplicates")
check(books_all.first3_last.notna().all(), "Step 2 read book list", "every serial has a chapter-3 time", f"{int(books_all.first3_last.isna().sum())} missing")
check((books_all.first3_last >= books_all.first_release).all(), "Step 2 read book list", "chapter-3 time is not before the first release", "some serials have chapter 3 before the first release")
print("Chapter-3 time from:", FIRST3_SOURCE)
print("First release range:", books_all.first_release.min(), "→", books_all.first_release.max())
print("Serials per year:", books_all.year.value_counts().sort_index().to_dict())

def stratified_sample(b, n, seed):
    """Sample in proportion to year, at least 15 serials per year."""
    share = b.year.value_counts(normalize=True).sort_index()
    k = (share * n).round().astype(int).clip(lower=15)
    while k.sum() > n: k[k.idxmax()] -= 1
    while k.sum() < n: k[k.idxmax()] += 1
    parts = [b[b.year == y].sample(min(int(c), int((b.year == y).sum())), random_state=seed) for y, c in k.items()]
    return pd.concat(parts)

if DRY_RUN:
    books = stratified_sample(books_all, min(DRY_N, len(books_all)), SEED)
else:
    books = books_all
books = books.sort_values("fiction_id").reset_index(drop=True)
N_BOOKS = len(books)
check(books.fiction_id.is_unique and N_BOOKS == (min(DRY_N, EXPECTED_N) if DRY_RUN else EXPECTED_N), "Step 2 read book list",
      "number of serials in this run is right and unique", f"{N_BOOKS} serials")
stage_end("Step 2 read book list", _t, f"{N_BOOKS} serials in this run")

[2026-11-14 22:00:33 UTC] === start: Step 2 read book list ===
All fields in the book list: ['fiction_id', 'first_release', 'n_chapters', 'title', 'author', 'status', 'is_original', 'tags', 'warnings', 'description', 'cover_url', 'followers', 'total_views', 'average_views', 'favorites', 'ratings', 'author_url', 'author_id', 'year', 'quarter', 'pct_in_quarter', 'top10', 'log_followers', 'first3_tokens', 'first3_truncated', 'first3_span_days']
Chapter-3 time from: first_release + first3_span_days (books_meta has no first3_last)
First release range: 2016-01-01 06:43:23+00:00 → 2024-12-31 20:31:04+00:00
Serials per year: {2016: 2205, 2017: 2503, 2018: 2877, 2019: 2913, 2020: 4036, 2021: 4771, 2022: 5700, 2023: 8075, 2024: 12165}
[2026-11-14 22:00:38 UTC] === done: Step 2 read book list (0.1 min) 45245 serials in this run ===


Step 3: assign cover status (no download yet).

Rules: custom cover = `cover_url` contains `/covers-` and not `nocover`; upload time = `time=` in the URL (Unix seconds, UTC);
upload time after the chapter-3 time → `after_ch3`; no grace period.
One more rule after download: if the image file's server modification time (Last-Modified) is after the data snapshot date → `image_changed`,
meaning the author replaced the cover after the snapshot, so the downloaded image is not the one from then; not valid.
Snapshot date = the latest `time=` in the book list (every cover change updates `time=`, so no upload in the snapshot is later than this day).
Last-Modified after `time=` but before the snapshot date means the site re-saved the file (an author's cover change also changes `time=`); the image is unchanged, so it stays valid.

In [4]:
_t = stage_start("Step 3 cover status")
_EPOCH = pd.Timestamp("1970-01-01", tz="UTC")
TS_MIN, TS_MAX = 1.2e9, 1.9e9          # only values between 2008 and 2030 count as valid Unix seconds

def judge(b):
    """Uses only the URL and times. pre is one of: no_custom_cover / no_timestamp / after_ch3 / candidate."""
    url = b.cover_url.fillna("").astype(str)
    low = url.str.lower()
    custom = low.str.contains("/covers-", regex=False) & ~low.str.contains("nocover", regex=False)
    cover_ts = pd.to_numeric(url.str.extract(r"[?&]time=(\d+)", expand=False), errors="coerce")
    cover_ts = cover_ts.where(cover_ts.between(TS_MIN, TS_MAX))      # values that do not look like Unix seconds count as no time
    ch3_ts = (b.first3_last - _EPOCH).dt.total_seconds()
    lag = (cover_ts - ch3_ts) / 86400.0
    pre = np.select([~custom, cover_ts.isna(), lag > 0], ["no_custom_cover", "no_timestamp", "after_ch3"], "candidate")
    out = b[["fiction_id", "year", "quarter", "top10", "author_id"]].copy()
    out["url"] = np.where(url.str.startswith("/"), "https://www.royalroad.com" + url, url)
    out["is_custom"] = custom.values
    out["cover_ts"] = np.where(custom, cover_ts, np.nan)
    out["lag_to_ch3_days"] = np.where(custom, lag, np.nan)       # upload time minus chapter-3 time; negative = before chapter 3
    out["pre"] = pre
    return out

pre = judge(books)
_all_ts = pd.to_numeric(books_all.cover_url.fillna("").astype(str).str.extract(r"[?&]time=(\d+)", expand=False), errors="coerce")
SNAPSHOT = pd.to_datetime(_all_ts[_all_ts.between(TS_MIN, TS_MAX)].max(), unit="s", utc=True)   # computed on the full book list so trial and full runs agree
print("Data snapshot date (latest time= in the book list):", SNAPSHOT)
check(len(pre) == N_BOOKS and pre.pre.notna().all(), "Step 3 cover status", f"each of {N_BOOKS} serials has one status", f"{len(pre)} rows")

# Check the URL rules on the data: count each of the three URL types and print a few examples
_low = books.cover_url.fillna("").astype(str).str.lower()
_kind = np.select([_low.str.contains("nocover", regex=False), _low.str.contains("/covers-", regex=False)], ["has nocover", "has /covers-"], "other")
URL_KINDS = pd.Series(_kind).value_counts().to_dict()
print("URL types:", URL_KINDS)
for k in ["has nocover", "has /covers-", "other"]:
    print(f"  examples of {k}:", books.cover_url[_kind == k].head(3).tolist())
_ts = pre.cover_ts.dropna()
if len(_ts):
    print("Upload time range:", pd.to_datetime(_ts.min(), unit="s", utc=True), "→", pd.to_datetime(_ts.max(), unit="s", utc=True))
_has_time = books.cover_url.fillna("").astype(str).str.contains(r"[?&]time=\d", regex=True) & pre.is_custom.values
_bad_time = int(_has_time.sum() - pre.cover_ts.notna().sum())
check(_bad_time <= 0.01 * max(int(_has_time.sum()), 1), "Step 3 cover status", "time= values are almost all Unix seconds between 2008 and 2030",
      f"{_bad_time}/{int(_has_time.sum())} are outside this range (maybe milliseconds or something else)")
print("\nStatus before download (candidate = becomes valid or download_failed after download):")
print(pd.crosstab(pre.year, pre.pre, margins=True))
stage_end("Step 3 cover status", _t)

[2026-11-14 22:00:38 UTC] === start: Step 3 cover status ===
Data snapshot date (latest time= in the book list): 2025-06-15 04:47:56+00:00
URL types: {'has /covers-': 36941, 'has nocover': 8302, 'other': 2}
  examples of has nocover: ['/dist/img/nocover-new-min.png', '/dist/img/nocover-new-min.png', '/dist/img/nocover-new-min.png']
  examples of has /covers-: ['https://www.royalroadcdn.com/public/covers-full/108-road-to-power.jpg', 'https://www.royalroadcdn.com/public/covers-full/352-irregular-elementalist.jpg', 'https://www.royalroadcdn.com/public/covers-large/637-a-tower-of-dreams.jpg?time=1693116962']
  examples of other: ['https://i.imgur.com/7G0Xgpx.png', 'https://www.royalroadcdn.com/lunnear/im-too-tired-of-being-op-so-ive-decided-to-lives-full-AAAAP8EwGQw=.jpg']
Upload time range: 2020-04-18 06:06:48+00:00 → 2025-06-15 04:47:56+00:00

Status before download (candidate = becomes valid or download_failed after download):
pre   after_ch3  candidate  no_custom_cover  no_timestamp   

Step 4: download and normalize images.

Download everything except `no_custom_cover`. Each image: convert to RGB → pad to a square with the average edge color (no cropping) → 384×384 JPEG (quality 90).

In [5]:
_local = threading.local()

def _session():
    if not hasattr(_local, "s"):
        _local.s = requests.Session()
        _local.s.headers.update({"User-Agent": USER_AGENT})
    return _local.s

def http_get(url):
    """Returns (status code, content, Last-Modified). Never raises; retries twice on failure; backs off on 429. Status -1 = timeout or network error."""
    last = (-1, None, None)
    for attempt in range(3):
        try:
            r = _session().get(url, timeout=TIMEOUT)
            lm = r.headers.get("Last-Modified")
            if r.status_code == 200 and r.content:
                return 200, r.content, lm
            last = (int(r.status_code), None, lm)
            if r.status_code in (404, 410):
                return last
            if r.status_code == 429:
                wait = r.headers.get("Retry-After", "")
                time.sleep(min(float(wait) if wait.isdigit() else 10 * (attempt + 1), 60))
            else:
                time.sleep(2 * (attempt + 1))
        except Exception:
            last = (-1, None, None)
            time.sleep(2 * (attempt + 1))
    return last

def content_box(w, h, size=IMG_SIZE):
    """Position (x0, y0, width, height) of the resized image on the square canvas."""
    s = size / max(w, h)
    nw, nh = max(1, round(w * s)), max(1, round(h * s))
    return (size - nw) // 2, (size - nh) // 2, nw, nh

def normalize_image(raw):
    """Returns (original width, original height, pHash, normalized JPEG bytes). Raises if the image cannot be opened."""
    im = Image.open(io.BytesIO(raw))
    im.seek(0)                                          # first frame of animated images
    if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
        im = im.convert("RGBA")
        bg = Image.new("RGB", im.size, (255, 255, 255))  # white background for transparency
        bg.paste(im, mask=im.split()[-1])
        im = bg
    else:
        im = im.convert("RGB")
    w, h = im.size
    if min(w, h) < 8:
        raise ValueError("image too small")
    ph = str(imagehash.phash(im))                        # computed on the original image, before padding
    a = np.asarray(im)
    ring = np.concatenate([a[:2].reshape(-1, 3), a[-2:].reshape(-1, 3), a[:, :2].reshape(-1, 3), a[:, -2:].reshape(-1, 3)])
    fill = tuple(int(round(x)) for x in ring.mean(0))    # average color of the outer 2-pixel ring
    x0, y0, nw, nh = content_box(w, h)
    canvas = Image.new("RGB", (IMG_SIZE, IMG_SIZE), fill)
    canvas.paste(im.resize((nw, nh), Image.LANCZOS), (x0, y0))
    buf = io.BytesIO()
    canvas.save(buf, "JPEG", quality=90)
    return w, h, ph, buf.getvalue()

def fetch_one(args):
    fid, url = args
    code, raw, lm = http_get(url)
    row = {"fiction_id": fid, "http_status": code, "last_modified": lm, "error": "", "img_w": None, "img_h": None, "phash": None, "jpg": None}
    if raw is None:
        row["error"] = "http"
        return row
    try:
        row["img_w"], row["img_h"], row["phash"], row["jpg"] = normalize_image(raw)
    except Exception as e:
        row["error"] = "decode:" + type(e).__name__
    return row

PART_COLS = ["fiction_id", "http_status", "last_modified", "error", "img_w", "img_h", "phash", "jpg"]
PART_PREFIX = f"part_v{STEP_VER['s4_download']}_"

def part_paths():
    return sorted(D_PARTS.glob(PART_PREFIX + "*.parquet"))

In [6]:
targets = pre.loc[pre.pre != "no_custom_cover", ["fiction_id", "url"]].sort_values("fiction_id").reset_index(drop=True)
N_TARGET = len(targets)
_n_parts = (N_TARGET + PART_SIZE - 1) // PART_SIZE

if need("s4_download", [P_IMG]):
    _t = stage_start("Step 4 download", total=N_TARGET)
    _done = _err = 0
    for i in range(_n_parts):
        p = D_PARTS / f"{PART_PREFIX}{i:05d}.parquet"
        chunk = targets.iloc[i * PART_SIZE:(i + 1) * PART_SIZE]
        if p.exists():
            d = load_df(p, ["fiction_id", "error"])
            if len(d) == len(chunk):
                _done += len(d); _err += int((d.error != "").sum())
                continue
        with ThreadPoolExecutor(N_THREADS) as ex:
            rows = list(ex.map(fetch_one, zip(chunk.fiction_id.tolist(), chunk.url.tolist())))
        d = pd.DataFrame(rows, columns=PART_COLS)
        d["img_w"] = d.img_w.astype("float64"); d["img_h"] = d.img_h.astype("float64")
        d["last_modified"] = d.last_modified.astype(object); d["phash"] = d.phash.astype(object)
        save_df(d, p)
        _done += len(d); _err += int((d.error != "").sum())
        progress(done=_done, total=N_TARGET, errors=_err)
        log(f"Download part {i + 1}/{_n_parts}: {_done}/{N_TARGET} so far, {_err} failed")
        if i == 0:
            check((d.error == "").any(), "Step 4 download", "at least one image in the first part downloads",
                  f"all failed, status codes {d.http_status.value_counts().to_dict()} (maybe blocked by the site)")
    check(_done == N_TARGET, "Step 4 download", f"{N_TARGET} serials processed", f"{_done} serials")
    _n_img = write_images(P_IMG, part_paths())
    check(_n_img == _done - _err and _n_img > 0, "Step 4 download", f"merged images = successful downloads {_done - _err}", f"{_n_img}")
    mark("s4_download")
    stage_end("Step 4 download", _t, f"{_n_img} succeeded, {_err} failed")

[2026-11-14 22:00:40 UTC] s4_download: output exists, skipping


Step 5: status table and duplicate detection.

In [7]:
if need("s5_status", [P_STATUS, P_SPOT]):
    _t = stage_start("Step 5 status table and duplicates")
    _pp = part_paths()
    check(len(_pp) == _n_parts, "Step 5 status table", f"{_n_parts} download parts (_download_parts must not be deleted before this step finishes)", f"{len(_pp)}")
    dl = pd.concat([load_df(p, PART_COLS[:-1]) for p in _pp], ignore_index=True)
    check(dl.fiction_id.is_unique and len(dl) == N_TARGET, "Step 5 status table", f"{N_TARGET} unique download records", f"{len(dl)} rows")
    dl["downloaded"] = dl.error == ""

    cs = pre.merge(dl, on="fiction_id", how="left")
    cs["downloaded"] = cs.downloaded.fillna(False).astype(bool)
    cs["cover_time"] = pd.to_datetime(cs.cover_ts, unit="s", utc=True)
    cs["last_modified"] = pd.to_datetime(cs.last_modified, format="%a, %d %b %Y %H:%M:%S GMT", utc=True, errors="coerce")
    changed = (cs.last_modified > SNAPSHOT + pd.Timedelta(days=1)).fillna(False).astype(bool)   # image replaced after the snapshot
    cs["status"] = np.where(cs.pre == "candidate",
                            np.where(~cs.downloaded, "download_failed", np.where(changed, "image_changed", "valid")), cs.pre)
    cs["valid_plus1d"] = (cs.is_custom & cs.cover_ts.notna() & (cs.lag_to_ch3_days <= 1) & cs.downloaded & ~changed).astype(bool)

    # Duplicate / template covers: number of serials per pHash, and whether different authors share it
    g = cs[cs.downloaded].groupby("phash")
    size, n_auth = g.fiction_id.transform("size"), g.author_id.transform("nunique")
    cs["dup_group_size"] = size.reindex(cs.index).astype("float64")
    cs["dup_cross_author"] = (n_auth.reindex(cs.index) > 1).fillna(False).astype(bool)

    cs = cs[["fiction_id", "year", "quarter", "top10", "status", "cover_time", "lag_to_ch3_days", "valid_plus1d",
             "http_status", "phash", "dup_group_size", "dup_cross_author", "img_w", "img_h", "downloaded", "last_modified"]]
    check(len(cs) == N_BOOKS and cs.fiction_id.is_unique, "Step 5 status table", f"{N_BOOKS} rows, unique fiction_id", f"{len(cs)} rows")
    check(cs.status.isin(STATUSES).all() and int(cs.status.value_counts().sum()) == N_BOOKS, "Step 5 status table",
          "exactly one valid status per row; counts sum to the total", cs.status.value_counts().to_dict())
    save_df(cs, P_STATUS)

    spot = cs.merge(books[["fiction_id", "cover_url", "first3_last"]], on="fiction_id")
    spot = spot.sample(min(20, len(spot)), random_state=SEED)[["fiction_id", "cover_url", "cover_time", "first3_last", "lag_to_ch3_days", "status"]]
    spot.to_csv(P_SPOT, index=False)
    mark("s5_status")
    stage_end("Step 5 status table and duplicates", _t)

cs = load_df(P_STATUS)
print(pd.crosstab(cs.year, cs.status, margins=True))

[2026-11-14 22:00:40 UTC] === start: Step 5 status table and duplicates ===


/tmp/ipykernel_6286/2909043628.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cs["downloaded"] = cs.downloaded.fillna(False).astype(bool)


[2026-11-14 22:01:11 UTC] === done: Step 5 status table and duplicates (0.5 min)  ===
status  after_ch3  image_changed  no_custom_cover  no_timestamp  valid    All
year                                                                         
2016           21              0             1270           913      1   2205
2017           46              0              890          1565      2   2503
2018          104              0              748          2017      8   2877
2019          225              0              646          2032     10   2913
2020         1027              2              702           698   1607   4036
2021         1397              8              750             7   2609   4771
2022         1760              4             1005             4   2927   5700
2023         2551             20              995             2   4507   8075
2024         3459             68             1298             5   7335  12165
All         10590            102             8304       

Step 6: extract embeddings (4 frozen models).

Every model gets the 384 square image resized straight to 224×224 (no center crop). CLIP vectors are L2-normalized; the others are left as is.

In [8]:
IMAGENET_NORM = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
CLIP_NORM = ([0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711])
HF_IDS = {"clip_b32": "openai/clip-vit-base-patch32", "clip_l14": "openai/clip-vit-large-patch14", "dinov2_base": "facebook/dinov2-base"}

class JpgDataset(torch.utils.data.Dataset):
    def __init__(self, jpgs): self.jpgs = jpgs
    def __len__(self): return len(self.jpgs)
    def __getitem__(self, i):
        im = Image.open(io.BytesIO(self.jpgs[i])).convert("RGB").resize((224, 224), Image.BICUBIC)
        return torch.from_numpy(np.asarray(im).copy()).permute(2, 0, 1)    # uint8, 3×224×224

def build_model(name):
    """Returns (model, forward function, (mean, std), use half precision, L2-normalize)."""
    if name == "resnet50":
        import torchvision
        m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = torch.nn.Identity()                                   # global pooling output, 2048-d
        return m, (lambda x: m(x)), IMAGENET_NORM, False, False
    if name in ("clip_b32", "clip_l14"):
        from transformers import CLIPVisionModelWithProjection
        m = CLIPVisionModelWithProjection.from_pretrained(HF_IDS[name])
        return m, (lambda x: m(pixel_values=x).image_embeds), CLIP_NORM, name == "clip_l14", True
    if name == "dinov2_base":
        from transformers import AutoModel
        m = AutoModel.from_pretrained(HF_IDS[name])
        return m, (lambda x: m(pixel_values=x).last_hidden_state[:, 0]), IMAGENET_NORM, True, False   # CLS
    raise ValueError(name)

def embed_images(name, jpgs, stage):
    m, fwd, (mean, std), half, l2 = build_model(name)
    m.eval().to(DEVICE)
    mean = torch.tensor(mean, device=DEVICE).view(1, 3, 1, 1)
    std = torch.tensor(std, device=DEVICE).view(1, 3, 1, 1)
    loader = torch.utils.data.DataLoader(JpgDataset(jpgs), batch_size=64, shuffle=False, num_workers=2)
    bs, outs, done = 64, [], 0
    for xb in loader:
        xb = (xb.to(DEVICE).float().div(255) - mean) / std
        i = 0
        while i < len(xb):
            try:
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=(half and DEVICE == "cuda")):
                    o = fwd(xb[i:i + bs])
                outs.append(o.float().cpu())
                i += bs
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                check(bs > 16, f"Step 6 embeddings {name}", "GPU memory suffices after lowering the batch size to 16", "still out of memory at batch size 16")
                bs //= 2
                log(f"{name}: out of GPU memory, batch size lowered to {bs}")
        done += len(xb)
        if (done // 64) % 50 == 0:
            progress(stage=stage, done=done, total=len(jpgs))
    E = torch.cat(outs).numpy().astype(np.float32)
    if l2:
        E = E / np.linalg.norm(E, axis=1, keepdims=True)
    del m
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return E

In [9]:
N_DOWNLOADED = int(cs.downloaded.sum())
for _name, _dim in MODELS.items():
    if need(f"s6_{_name}", [P_EMB[_name]]):
        _stage = f"Step 6 embeddings {_name}"
        _ids, _jpgs = read_images()
        _t = stage_start(_stage, total=len(_jpgs))
        E = embed_images(_name, _jpgs, _stage)
        check(E.shape == (len(_ids), _dim), _stage, f"embedding shape ({len(_ids)}, {_dim})", f"{E.shape}")
        check(bool(np.isfinite(E).all()), _stage, "no missing or infinite values in the embeddings", f"{int((~np.isfinite(E)).sum())} bad values")
        check(float(E.std(0).mean()) > 1e-6, _stage, "different images give different vectors", "all vectors are nearly identical (model or input problem)")
        write_emb(P_EMB[_name], _ids, E)
        mark(f"s6_{_name}")
        stage_end(_stage, _t, f"{E.shape}")
        del E

[2026-11-14 22:01:11 UTC] s6_resnet50: output exists, skipping
[2026-11-14 22:01:11 UTC] s6_clip_b32: output exists, skipping
[2026-11-14 22:01:12 UTC] s6_clip_l14: output exists, skipping
[2026-11-14 22:01:12 UTC] s6_dinov2_base: output exists, skipping


Step 7: basic visual statistics.

Computed on the cover area only (no padding). `text_area_proxy` uses no OCR; it is the share of small blocks dense with fine edges, a rough estimate.

In [10]:
def visual_stats(jpg, w, h):
    x0, y0, nw, nh = content_box(int(w), int(h))
    a = np.asarray(Image.open(io.BytesIO(jpg)).convert("RGB"))[y0:y0 + nh, x0:x0 + nw].astype(np.float32) / 255.0
    r, g, b = a[..., 0], a[..., 1], a[..., 2]
    luma = 0.299 * r + 0.587 * g + 0.114 * b
    mx, mn = a.max(-1), a.min(-1)
    sat = np.where(mx > 0, (mx - mn) / np.maximum(mx, 1e-6), 0)
    rg, yb = r - g, 0.5 * (r + g) - b                                  # Hasler–Süsstrunk colorfulness
    colorful = np.sqrt(rg.std() ** 2 + yb.std() ** 2) + 0.3 * np.sqrt(rg.mean() ** 2 + yb.mean() ** 2)
    gx, gy = np.zeros_like(luma), np.zeros_like(luma)
    gx[:, 1:-1] = luma[:, 2:] - luma[:, :-2]
    gy[1:-1, :] = luma[2:, :] - luma[:-2, :]
    edge = np.hypot(gx, gy) > 0.2
    bh, bw = edge.shape[0] // 8, edge.shape[1] // 8
    busy = edge[:bh * 8, :bw * 8].reshape(bh, 8, bw, 8).mean((1, 3)) > 0.25 if bh and bw else np.zeros((1, 1))
    return {"brightness": luma.mean(), "contrast": luma.std(), "saturation": sat.mean(), "colorfulness": colorful,
            "edge_density": edge.mean(), "text_area_proxy": busy.mean(), "dark_frac": (luma < 0.2).mean(),
            "bright_frac": (luma > 0.8).mean(), "mean_r": r.mean(), "mean_g": g.mean(), "mean_b": b.mean(), "aspect_w_over_h": w / h}

if need("s7_stats", [P_STATS]):
    _ids, _jpgs = read_images()
    _t = stage_start("Step 7 visual statistics", total=len(_jpgs))
    _wh = cs.set_index("fiction_id")[["img_w", "img_h"]].loc[_ids]
    rows = []
    for j, (jpg, w, h) in enumerate(zip(_jpgs, _wh.img_w.values, _wh.img_h.values)):
        rows.append(visual_stats(jpg, w, h))
        if (j + 1) % 5000 == 0:
            progress(done=j + 1)
    stats = pd.DataFrame(rows).astype(np.float32)
    stats.insert(0, "fiction_id", _ids)
    check(len(stats) == len(_ids) and bool(np.isfinite(stats.drop(columns="fiction_id").values).all()), "Step 7 visual statistics",
          f"{len(_ids)} rows, no missing values", f"{len(stats)} rows, {int(stats.isna().sum().sum())} missing")
    save_df(stats, P_STATS)
    mark("s7_stats")
    stage_end("Step 7 visual statistics", _t)

[2026-11-14 22:01:12 UTC] s7_stats: output exists, skipping


Step 8: self-check (not model selection).

For each embedding: valid serials only, a logistic regression trained on 2017–21 and validated on 2022. 2023 is not touched.
Two controls: year and quarter only; shuffled training labels (should be near 0.5, catches misaligned rows). Trial-run samples are too small; do not read the numbers.

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

def _fit_eval(Xtr, ytr, Xva, yva):
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=2000))   # scaling fitted on training years only; C fixed, not tuned
    s = clf.fit(Xtr, ytr).decision_function(Xva)
    return {"auc": float(roc_auc_score(yva, s)), "ap": float(average_precision_score(yva, s))}

P_SMOKE = D_OUT / "smoke_test.json"
if need("s8_selfcheck", [P_SMOKE]):
    _t = stage_start("Step 8 self-check")
    v = cs[cs.status == "valid"].set_index("fiction_id")
    smoke = {"note": "valid serials only; train 2017–2021, validate 2022; 2023 unused. Logistic regression, C=0.01, not tuned.", "models": {}}
    for _name in list(MODELS) + ["stats"]:
        if _name == "stats":
            _s = load_df(P_STATS); ids, E = _s.fiction_id.values, _s.drop(columns="fiction_id").values
        else:
            ids, E = read_emb(P_EMB[_name])
        keep = np.isin(ids, v.index.values)
        ids, E = ids[keep], E[keep]
        yr, y = v.year.loc[ids].values, v.top10.loc[ids].values.astype(int)
        qn = pd.Series(v.quarter.loc[ids].values).str[-1].astype(int).values
        tr, va = (yr >= TRAIN_YEARS[0]) & (yr <= TRAIN_YEARS[1]), yr == VAL_YEAR
        res = {"n_train": int(tr.sum()), "n_val": int(va.sum()), "pos_train": int(y[tr].sum()), "pos_val": int(y[va].sum())}
        if min(res["n_train"], res["n_val"]) < 20 or min(res["pos_train"], res["pos_val"]) < 2 or res["pos_val"] == res["n_val"] or res["pos_train"] == res["n_train"]:
            res["skipped"] = "sample too small, skipped"
        else:
            res.update(_fit_eval(E[tr], y[tr], E[va], y[va]))
            res["shuffled_label_auc"] = _fit_eval(E[tr], np.random.default_rng(SEED).permutation(y[tr]), E[va], y[va])["auc"]
            if "year_quarter_only" not in smoke:
                Z = np.column_stack([yr] + [(qn == k).astype(float) for k in (1, 2, 3, 4)])
                smoke["year_quarter_only"] = _fit_eval(Z[tr], y[tr], Z[va], y[va])
        smoke["models"][_name] = res
        log(f"Self-check {_name}: {res}")
    P_SMOKE.write_text(json.dumps(smoke, ensure_ascii=False, indent=1), encoding="utf-8")
    mark("s8_selfcheck")
    stage_end("Step 8 self-check", _t)
smoke = json.loads(P_SMOKE.read_text(encoding="utf-8"))

[2026-11-14 22:01:14 UTC] === start: Step 8 self-check ===
[2026-11-14 22:01:30 UTC] Self-check resnet50: {'n_train': 4236, 'n_val': 2927, 'pos_train': 327, 'pos_val': 173, 'auc': 0.5630947733407214, 'ap': 0.07612374832910951, 'shuffled_label_auc': 0.5088972004986966}
[2026-11-14 22:01:33 UTC] Self-check clip_b32: {'n_train': 4236, 'n_val': 2927, 'pos_train': 327, 'pos_val': 173, 'auc': 0.6146540397362114, 'ap': 0.09366492815818775, 'shuffled_label_auc': 0.4854714739674504}
[2026-11-14 22:01:39 UTC] Self-check clip_l14: {'n_train': 4236, 'n_val': 2927, 'pos_train': 327, 'pos_val': 173, 'auc': 0.6743654001956167, 'ap': 0.1346761495449303, 'shuffled_label_auc': 0.4654711381448319}
[2026-11-14 22:01:44 UTC] Self-check dinov2_base: {'n_train': 4236, 'n_val': 2927, 'pos_train': 327, 'pos_val': 173, 'auc': 0.5725691689649526, 'ap': 0.07700444389225797, 'shuffled_label_auc': 0.541406928860176}
[2026-11-14 22:01:46 UTC] Self-check stats: {'n_train': 4236, 'n_val': 2927, 'pos_train': 327, 'pos_

Step 9: exploration (descriptive only, not used in models).

Clustering uses CLIP ViT-L/14 vectors of `valid` serials only; fitted on 2017–2022, other years are only assigned.
No statistic that uses the `top10` label includes 2023.

In [12]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

LABEL_OK = lambda year: year != TEST_YEAR          # 2023 labels are not used in any statistic
PERIODS = {"2017_2022": (2017, 2022), "2024": (2024, 2024)}   # lift on training + validation years combined; 2024 shown separately; 2023 not looked at

def contact_sheet(jpgs, path, cols=8, thumb=128):
    rows = (len(jpgs) + cols - 1) // cols
    sheet = Image.new("RGB", (cols * thumb, max(rows, 1) * thumb), (255, 255, 255))
    for i, j in enumerate(jpgs):
        sheet.paste(Image.open(io.BytesIO(j)).convert("RGB").resize((thumb, thumb), Image.BICUBIC), ((i % cols) * thumb, (i // cols) * thumb))
    sheet.save(path)

P_CSUM, P_SBY, P_SENS, P_DUP = D_OUT / "clusters_summary.csv", D_OUT / "status_by_year.csv", D_OUT / "sensitivity_plus1d.csv", D_OUT / "dup_summary.csv"
if need("s9_explore", [P_CLUST, P_CSUM, P_SBY, P_SENS, P_DUP]):
    _t = stage_start("Step 9 exploration")
    for f in D_FIG.glob("*.png"):
        f.unlink()

    # ---- 9a counts and hit rates by year and status (no hit rate shown for 2023) ----
    g = cs.groupby(["year", "status"]).agg(n=("fiction_id", "size"), top10_rate=("top10", "mean")).reset_index()
    g["share_of_year"] = g.n / g.groupby("year").n.transform("sum")
    g.loc[~LABEL_OK(g.year), "top10_rate"] = np.nan
    pool = cs[cs.year.between(TRAIN_YEARS[0], VAL_YEAR)]
    gp = pool.groupby("status").agg(n=("fiction_id", "size"), top10_rate=("top10", "mean")).reset_index()
    gp["share_of_year"] = gp.n / len(pool) if len(pool) else np.nan
    gp.insert(0, "year", "2017-2022")
    g["year"] = g.year.astype(str)
    pd.concat([g, gp], ignore_index=True)[["year", "status", "n", "share_of_year", "top10_rate"]].to_csv(P_SBY, index=False)

    ct = pd.crosstab(cs.year, cs.status).reindex(columns=STATUSES, fill_value=0)
    ax = ct.div(ct.sum(axis=1), axis=0).plot(kind="bar", stacked=True, figsize=(9, 4.5), width=0.8)
    ax.set_ylabel("share of books"); ax.set_title("Cover status by release year"); ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout(); plt.savefig(D_FIG / "status_by_year.png", dpi=120); plt.close()

    if len(gp):
        ax = gp.set_index("status").top10_rate.reindex(STATUSES).plot(kind="bar", figsize=(7, 4), color="#4c72b0")
        ax.set_ylabel("top10 rate"); ax.set_title("Top-10% rate by cover status (2017-2022 pooled)")
        plt.tight_layout(); plt.savefig(D_FIG / "top10_by_status.png", dpi=120); plt.close()

    lag = cs.lag_to_ch3_days.dropna()
    if len(lag):
        plt.figure(figsize=(8, 4)); plt.hist(lag.clip(-10, 30), bins=160, color="#4c72b0")
        plt.axvline(0, color="red", lw=1); plt.xlabel("cover upload time minus chapter-3 time (days, clipped to [-10, 30])"); plt.ylabel("books")
        plt.title("When was the current cover uploaded?"); plt.tight_layout(); plt.savefig(D_FIG / "lag_hist.png", dpi=120); plt.close()

    # ---- 9b sensitivity: valid count with a 1-day grace period ----
    s = cs.groupby("year").agg(n_books=("fiction_id", "size"), n_valid=("status", lambda x: int((x == "valid").sum())),
                               n_valid_plus1d=("valid_plus1d", "sum")).reset_index()
    s["added_by_1d_grace"] = s.n_valid_plus1d - s.n_valid
    tot = s.drop(columns="year").sum().to_frame().T; tot.insert(0, "year", "ALL")
    s["year"] = s.year.astype(str)
    pd.concat([s, tot], ignore_index=True).to_csv(P_SENS, index=False)

    # ---- 9c duplicate cover summary ----
    d = cs[cs.downloaded]
    grp = d.groupby("phash").agg(size=("fiction_id", "size"), cross=("dup_cross_author", "max")).reset_index()
    grp["bucket"] = pd.cut(grp["size"], [0, 1, 2, 5, 20, 10 ** 9], labels=["1 (unique)", "2", "3-5", "6-20", "21+"])
    dup = grp.groupby("bucket", observed=False).agg(n_groups=("size", "size"), n_books=("size", "sum"),
                                                     n_groups_cross_author=("cross", "sum")).reset_index().rename(columns={"bucket": "dup_group_size"})
    dup.to_csv(P_DUP, index=False)

    # ---- 9d clustering ----
    ids, E = read_emb(P_EMB["clip_l14"])
    v = cs[cs.status == "valid"].set_index("fiction_id")
    keep = np.isin(ids, v.index.values)
    ids, E = ids[keep], E[keep]
    yr, y = v.year.loc[ids].values, v.top10.loc[ids].values.astype(float)
    fit = (yr >= TRAIN_YEARS[0]) & (yr <= VAL_YEAR)
    n_fit = int(fit.sum())
    sil = {}
    if n_fit < 16:
        log(f"Clustering: only {n_fit} valid serials in 2017–2022, too few, skipping")
        labels = np.zeros(len(ids), dtype=int); K = 1
    else:
        ks = list(range(6, min(20, n_fit // 8) + 1)) or list(range(2, max(2, n_fit // 8) + 1))
        rng = np.random.default_rng(SEED)
        sub = rng.choice(n_fit, min(n_fit, 10000), replace=False)
        Xf = E[fit]
        for k in ks:
            km = KMeans(k, n_init=5, random_state=SEED).fit(Xf)
            sil[k] = float(silhouette_score(Xf[sub], km.labels_[sub], metric="cosine"))
            progress(done=k, total=ks[-1])
        K = max(sil, key=sil.get)
        km = KMeans(K, n_init=10, random_state=SEED).fit(Xf)
        labels = km.predict(E)
        log(f"Clustering: silhouette {json.dumps({k: round(s_, 4) for k, s_ in sil.items()})} → k = {K}")
        plt.figure(figsize=(6, 3.5)); plt.plot(list(sil), list(sil.values()), "o-"); plt.axvline(K, color="red", lw=1)
        plt.xlabel("k"); plt.ylabel("silhouette (cosine)"); plt.title("k selection, CLIP ViT-L/14, valid covers 2017-2022")
        plt.tight_layout(); plt.savefig(D_FIG / "silhouette.png", dpi=120); plt.close()
        # Representative images per cluster: the 24 closest to the cluster center
        _all_ids, _all_jpgs = read_images()
        _pos = pd.Series(np.arange(len(_all_ids)), index=_all_ids)
        for c in range(K):
            m = np.where(labels == c)[0]
            near = m[np.argsort(((E[m] - km.cluster_centers_[c]) ** 2).sum(1))[:24]]
            contact_sheet([_all_jpgs[_pos[i]] for i in ids[near]], D_FIG / f"cluster_{c:02d}.png")
    save_df(pd.DataFrame({"fiction_id": ids, "cluster": labels.astype(np.int32)}), P_CLUST)

    # Per cluster: count, share by year (of that year's valid serials), hit rate and lift by period
    cl = pd.DataFrame({"cluster": labels, "year": yr, "top10": y})
    summ = cl.groupby("cluster").size().rename("n_valid").to_frame()
    summ["share_all"] = summ.n_valid / len(cl)
    for yv in sorted(cl.year.unique()):
        suby = cl[cl.year == yv]
        summ[f"share_{yv}"] = suby.groupby("cluster").size().reindex(summ.index, fill_value=0) / max(len(suby), 1)
    for pname, (a, b_) in PERIODS.items():
        sub_ = cl[cl.year.between(a, b_)]                                   # never includes 2023
        base = sub_.top10.mean() if len(sub_) else np.nan
        gg = sub_.groupby("cluster").top10.agg(["size", "sum", "mean"]).reindex(summ.index)
        summ[f"n_{pname}"], summ[f"n_top10_{pname}"] = gg["size"].fillna(0).astype(int), gg["sum"].fillna(0).astype(int)
        summ[f"top10_rate_{pname}"], summ[f"lift_{pname}"] = gg["mean"], gg["mean"] / base if base else np.nan
    summ["k_selected"], summ["silhouette_at_k"] = K, sil.get(K, np.nan)
    summ.reset_index().to_csv(P_CSUM, index=False)
    mark("s9_explore")
    stage_end("Step 9 exploration", _t, f"k = {K}")

[2026-11-14 22:01:46 UTC] === start: Step 9 exploration ===
[2026-11-14 22:03:02 UTC] Clustering: silhouette {"6": 0.0202, "7": 0.0121, "8": 0.0062, "9": 0.0034, "10": 0.0016, "11": 0.0019, "12": -0.0118, "13": 0.002, "14": -0.0126, "15": -0.012, "16": -0.0093, "17": -0.0114, "18": -0.0172, "19": -0.017, "20": -0.016} → k = 6
[2026-11-14 22:03:34 UTC] === done: Step 9 exploration (1.8 min) k = 6 ===


Step 10: export and acceptance checks.

In [ ]:
_t = stage_start("Step 10 export and acceptance")
cs = load_df(P_STATUS)
CHECKS = []
def add_check(name, ok, detail):
    CHECKS.append({"check": name, "result": "PASS" if ok is True else ("FAIL" if ok is False else str(ok)), "detail": str(detail)})
    print(("PASS  " if ok is True else "FAIL  " if ok is False else f"{ok}  ") + name + " | " + str(detail))

# Loader for teammates (the README example is this function); run once here to make sure it works
def load_cover_block(fiction_ids, status_path, emb_path):
    """Returns (V, cover_valid). Rows of V follow fiction_ids; rows without a valid cover are all 0 and cover_valid is False."""
    st = pd.read_parquet(status_path, columns=["fiction_id", "status"])
    t = pq.read_table(emb_path)
    ids = t.column("fiction_id").to_numpy()
    E = t.column("emb").combine_chunks().flatten().to_numpy().reshape(len(ids), -1).astype(np.float32)
    fiction_ids = np.asarray(fiction_ids)
    cover_valid = np.isin(fiction_ids, st.fiction_id[st.status == "valid"].values)
    pos = pd.Series(np.arange(len(ids)), index=ids).reindex(fiction_ids[cover_valid]).values.astype(int)
    V = np.zeros((len(fiction_ids), E.shape[1]), dtype=np.float32)
    V[cover_valid] = E[pos]
    return V, cover_valid

counts = cs.status.value_counts().reindex(STATUSES, fill_value=0).astype(int).to_dict()
n_valid, n_dl = counts["valid"], int(cs.downloaded.sum())

# 1 row counts and status
add_check("status table rows = book list rows", len(cs) == N_BOOKS and (DRY_RUN or len(cs) == EXPECTED_N), f"{len(cs)} rows, book list {N_BOOKS} rows")
add_check("each serial has exactly one status; counts sum to the total", bool(cs.fiction_id.is_unique and cs.status.isin(STATUSES).all() and sum(counts.values()) == N_BOOKS), counts)

# 2 re-derive status from the raw URLs with the section 2 rules and compare row by row
u = cs.merge(books[["fiction_id", "cover_url", "first3_last"]], on="fiction_id")
_url = u.cover_url.fillna("").astype(str); _lw = _url.str.lower()
_custom = _lw.str.contains("/covers-", regex=False) & ~_lw.str.contains("nocover", regex=False)
_ts = pd.to_numeric(_url.str.extract(r"[?&]time=(\d+)", expand=False), errors="coerce")
_ts = _ts.where(_ts.between(TS_MIN, TS_MAX))
_ch3 = (u.first3_last - _EPOCH).dt.total_seconds()
_chg = (u.last_modified > SNAPSHOT + pd.Timedelta(days=1)).fillna(False).astype(bool)
_exp = np.select([~_custom, _ts.isna(), _ts > _ch3, ~u.downloaded, _chg], STATUSES[:5], "valid")
add_check("status table matches the section 2 rules row by row", bool((_exp == u.status.values).all()), f"{int((_exp != u.status.values).sum())} rows differ")
_sby = pd.read_csv(P_SBY, dtype={"year": str})
_sby = _sby[_sby.year != "2017-2022"]
_a = _sby.set_index(["year", "status"]).n.sort_index()
_b = cs.assign(year=cs.year.astype(str)).groupby(["year", "status"]).size().sort_index()
add_check("status_by_year.csv matches the status table", {k: int(x) for k, x in _a.items()} == {k: int(x) for k, x in _b.items()}, f"{len(_a)} year × status combinations")
add_check("spot_check.csv exported for manual review", P_SPOT.exists() and len(pd.read_csv(P_SPOT)) == min(20, N_BOOKS), str(P_SPOT.name))

# 3 images and embeddings
_img_ids, _ = read_images()
add_check("serials with image files = serials downloaded", set(_img_ids.tolist()) == set(cs.fiction_id[cs.downloaded].tolist()), f"{len(_img_ids)} images")
for _name, _dim in MODELS.items():
    ids, E = read_emb(P_EMB[_name])
    ok = len(set(ids.tolist())) == len(ids) == n_dl and set(ids.tolist()) <= set(cs.fiction_id.tolist()) and E.shape[1] == _dim and bool(np.isfinite(E).all())
    add_check(f"embedding {_name}: serials are a subset of the status table, {_dim} dims, no missing values", ok, f"shape {E.shape}")
_s = load_df(P_STATS)
add_check("visual statistics: rows = successful downloads, no missing values", len(_s) == n_dl and not bool(_s.isna().any().any()), f"{_s.shape}")

# 4 download failure rate: 404/410 means the serial was taken down, counted separately; other failures (timeout, 5xx, 429, unreadable image) must stay under 5%
att = cs[cs.status != "no_custom_cover"]
gone = int(att.http_status.isin([404, 410]).sum())
other_fail = int((~att.downloaded).sum()) - gone
rate = other_fail / max(len(att), 1)
add_check("download failure rate at most 5% (excluding 404/410 takedowns)", rate <= 0.05,
          f"attempted {len(att)}, succeeded {n_dl}, taken down {gone} ({gone / max(len(att), 1):.1%}), other failures {other_fail} ({rate:.1%})")

# 5 self-check AUC (required for the full run only)
_auc = {k: r.get("auc") for k, r in smoke["models"].items() if k in MODELS}
if DRY_RUN:
    add_check("self-check AUC above 0.5", "n/a (trial run)", {k: (round(a, 3) if a is not None else "skipped") for k, a in _auc.items()})
else:
    add_check("self-check AUC above 0.5", all(a is not None and a > 0.5 for a in _auc.values()), {k: (round(a, 3) if a is not None else "skipped") for k, a in _auc.items()})

# 6 the README example code can load the V block
try:
    V, cv = load_cover_block(books_all.fiction_id.values, P_STATUS, P_EMB["clip_l14"])
    ok = V.shape == (len(books_all), MODELS["clip_l14"]) and int(cv.sum()) == n_valid and bool((np.abs(V[cv]).sum(1) > 0).all()) and bool((V[~cv] == 0).all())
    add_check("README example code loads the V block", ok, f"V shape {V.shape}, {int(cv.sum())} valid covers")
except Exception as e:
    add_check("README example code loads the V block", False, repr(e))

# For reference only (not PASS/FAIL): is time= the upload time? For valid serials, Last-Modified should be almost the same as time=
_v = cs[(cs.status == "valid") & cs.last_modified.notna()]
_n_chg = counts["image_changed"]
_den = counts["valid"] + _n_chg
add_check("info: share of candidates whose cover changed after the snapshot (image_changed)", "INFO", f"{_n_chg / max(_den, 1):.1%} ({_n_chg}/{_den}); snapshot date {SNAPSHOT:%Y-%m-%d}")
# Relationship between Last-Modified and time=, by year (serials with time= that downloaded), exported for review
_d = cs[cs.downloaded & cs.cover_time.notna()].copy()
_gapd = (_d.last_modified - _d.cover_time).dt.total_seconds() / 86400
_d["lm_kind"] = np.select([_d.last_modified.isna(), _gapd <= 1, _d.last_modified <= SNAPSHOT + pd.Timedelta(days=1)],
                          ["no_last_modified", "same_as_time", "resaved_before_snapshot"], "changed_after_snapshot")
_lm = pd.crosstab(_d.year, _d.lm_kind)
_lm.to_csv(D_OUT / "last_modified_check.csv")
_rs = _d[_d.lm_kind == "resaved_before_snapshot"].last_modified.dt.strftime("%Y-%m").value_counts().head(5).to_dict()
add_check("info: Last-Modified vs time= (by year in out/last_modified_check.csv)", "INFO",
          f"{_d.lm_kind.value_counts().to_dict()}; month with most re-saves {_rs}")
if len(_v):
    _gap = (_v.last_modified - _v.cover_time).dt.total_seconds() / 86400
    add_check("info: Last-Modified minus time= for valid serials (days)", "INFO",
              f"median {float(_gap.median()):.2f}, 90th percentile {float(_gap.quantile(.9)):.2f}, valid serials without Last-Modified {int((cs.status == 'valid').sum()) - len(_v)}")

n_fail = sum(c["result"] == "FAIL" for c in CHECKS)
summary = {"run": "dry_run" if DRY_RUN else "full", "finished": _now(), "n_books": N_BOOKS, "status_counts": counts,
           "n_downloaded": n_dl, "url_kinds": URL_KINDS, "first3_last_source": FIRST3_SOURCE,
           "n_fail": n_fail, "all_pass": n_fail == 0, "checks": CHECKS}
for p in [D_LOG / "checks.json", D_OUT / "checks.json"]:
    p.write_text(json.dumps(summary, ensure_ascii=False, indent=1), encoding="utf-8")

# ---- Export 05_for_github ----
_cands = [BASE / "cover_pipeline.ipynb", BASE.parent / "cover_pipeline.ipynb"]   # either cover_v2/ or RR_project1/ works
_nb = next((p for p in _cands if p.exists()), _cands[0])
if _nb.exists():
    shutil.copy(_nb, D_GH / "cover_pipeline.ipynb")
else:
    log(f"Note: no notebook found at {_nb}; 05_for_github has no notebook copy yet (put the notebook there and rerun this cell)")

_readme = D_GH / "README.md"
_DRAFT = "<!-- draft: auto-generated by cover_pipeline.ipynb -->"
if not _readme.exists() or _readme.read_text(encoding="utf-8").startswith(_DRAFT):
    _readme.write_text(f"""{_DRAFT}
# cover: cover vectors (V block)

Draft auto-generated by the notebook; the final version is written after the full run.

- Mode: {summary['run']}, serials {N_BOOKS}
- Counts by status: {counts}
- Chapter-3 time from: {FIRST3_SOURCE}
- Checks: {sum(c['result'] == 'PASS' for c in CHECKS)} passed / {n_fail} failed (see `out/checks.json`)
""", encoding="utf-8")

(BASE / "README_folders.txt").write_text("""RR_project1/cover_v2/ folder guide
dry_run/   trial run (300 serials), same layout as full/
full/      full run (all serials)
  00_input/        book list books_meta.parquet as downloaded from Hugging Face
  01_status/       cover_status.parquet (one row per serial: cover status; handoff), spot_check.csv (20 serials for manual review)
  02_images/       cover_images_384.parquet (normalized cover images), _download_parts/ (download parts; can be deleted once everything finishes)
  03_embeddings/   embeddings from 4 models cover_emb_*.parquet and visual statistics cover_stats.parquet (handoff)
  04_explore/      cover_clusters.parquet (clustering result, descriptive only)
  05_for_github/   whole folder goes into the team repo's cover/: README.md, notebook copy, out/ (summary tables, figures, checks.json)
  logs/            run_log.txt (run log), progress.json (current progress), checks.json (check results)
""", encoding="utf-8")

stage_end("Step 10 export and acceptance", _t)
progress(stage="all done", finished=True, failed=n_fail > 0,
         message=("all checks passed" if n_fail == 0 else f"{n_fail} checks failed, see logs/checks.json"))
log(f"All done: {sum(c['result'] == 'PASS' for c in CHECKS)} checks passed, {n_fail} failed.")
print("\nCounts by status:", counts)

[2026-11-14 22:03:34 UTC] === start: Step 10 export and acceptance ===
PASS  status table rows = book list rows | 45245 rows, book list 45245 rows
PASS  each serial has exactly one status; counts sum to the total | {'no_custom_cover': 8304, 'no_timestamp': 7243, 'after_ch3': 10590, 'download_failed': 0, 'image_changed': 102, 'valid': 19006}
PASS  status table matches the section 2 rules row by row | 0 rows differ
PASS  status_by_year.csv matches the status table | 41 year × status combinations
PASS  spot_check.csv exported for manual review | spot_check.csv
PASS  serials with image files = serials downloaded | 36938 images
